# Session 04: Modules, Packages, Classes and Testing

**Notebook 1 of 3: Concepts** · used during the live session and for revision afterwards (about 60 minutes on your own)

Your incident report works. It's also two files, about 400 lines, held together by dict keys typed as strings and tested by hand-written checker scripts. That's fine for a prototype and painful for a team. This session turns it into a **package**: modules with clear responsibilities, dataclasses instead of loose dicts, pytest tests, and Ruff configured for the project.

By the end of this notebook you will be able to:

1. **Explain** how `import` finds and loads modules (`sys.path`, `sys.modules`, packages), and **recognise** shadowing and circular imports.
2. **Organise** code into a `src/`-layout package with `__init__.py`, `__main__.py` and a command defined in `pyproject.toml`.
3. **Write** classes the Python way, and **model** domain objects with `@dataclass`, using composition rather than inheritance.
4. **Test** code with pytest: plain `assert`, fixtures, `tmp_path`, `parametrize` and `pytest.raises`.
5. **Configure** and run Ruff, and **debug** with `breakpoint()` and the VS Code debugger.

**Then:** `03-debugging-lab.ipynb` (broken imports and tests to fix), and the quiz in `02-practice-quiz.ipynb`.

> **Kernel and tools:** this notebook runs pytest and Ruff on small example projects that it creates in a temporary folder. Both come with the course environment's `dev` group, so run `uv sync` in the course repository first if you haven't since Session 03.

## The engineering problem

A colleague joins the Orbital Logistics (fictional) platform team to help with the incident report. Within a day they hit three problems:

1. They change `ticket["team"]` to `ticket["teams"]` in one function. Nothing complains until a nightly run fails with `KeyError: 'teams'`.
2. They want to reuse `read_export` in another tool, but importing `incident_report.py` also imports everything else, and running it as a script and importing it behave differently.
3. They fix a bug and aren't sure what else they broke, because the only tests are a custom script that prints `PASS`/`FAIL`.

Java and C# answer these with packages and namespaces, classes with declared fields, and JUnit or xUnit. Python has direct equivalents, with less ceremony: **modules and packages**, **dataclasses**, and **pytest**.

A small helper first. Several examples below write a tiny project to a temporary folder and run a command in it, so you see real output from real tools. You don't need to study this cell.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path


def write_files(root: Path, files: dict[str, str]) -> None:
    """Create each file (relative path -> source text) under root."""
    for relative_path, text in files.items():
        path = root / relative_path
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(textwrap.dedent(text).lstrip(), encoding="utf-8")


def run(*args: str, cwd: Path, env_path: str | None = None) -> int:
    """Run `python <args>` in cwd with this kernel's Python; print output, return exit code."""
    import os

    env = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")}
    env |= {"NO_COLOR": "1", "PY_COLORS": "0"}  # plain text: no colour codes in the notebook
    if env_path is not None:
        env["PYTHONPATH"] = env_path
    result = subprocess.run(
        [sys.executable, *args], cwd=cwd, capture_output=True, text=True, encoding="utf-8", env=env
    )
    print((result.stdout + result.stderr).rstrip())
    return result.returncode


print("helpers ready")

## Part 1: Modules and imports

A **module** is just a `.py` file. Its name is the file name without `.py`. You've been importing standard-library modules since Session 01; your own files work the same way.

Two facts from earlier sessions carry the whole idea:

- Importing a module **runs its top-level code once** and caches the module object in `sys.modules` (Session 01, Part 1). Later imports reuse it.
- `import` searches the directories on `sys.path` **in order** and takes the first match.

![How Python finds and loads a module](../../../assets/diagrams/python-module-imports.svg)

**How this works**

- `import incident_report.loading` first checks `sys.modules`. If the module was already imported anywhere in the process, Python reuses that object. That's why module-level code (opening a connection, reading config) runs only once.
- Otherwise Python walks `sys.path`. The first entry is the running script's folder (or the current folder for `python -m` and the REPL), then the standard library, then the virtual environment's `site-packages`, where `uv sync` installed your package.
- Because the script's folder comes first, your own `json.py` would hide the standard library's `json`. That's **shadowing**, and the debugging lab reproduces it.
- On the right, the package's modules import each other in one direction only: everything points towards `models`, which imports nothing from the package. That keeps imports predictable and rules out circular imports.
- `ModuleNotFoundError` simply means no `sys.path` entry contained the name. The fix is almost always "run with the right interpreter or environment" (Session 01), not "add a path by hand".

### `import x` versus `from x import y`

Let's build a two-module example and run it. **Predict:** how many times does `rules.py` print its "loading" message?

In [ ]:
with tempfile.TemporaryDirectory() as workdir:
    root = Path(workdir)
    write_files(root, {
        "rules.py": """
            print(f"loading rules (__name__ = {__name__!r})")

            SLA_HOURS = {"P1": 4, "P2": 24}


            def is_breached(priority, age_hours):
                return age_hours > SLA_HOURS[priority]
            """,
        "main.py": """
            import rules
            from rules import is_breached
            import rules as sla_rules

            print(rules.is_breached("P1", 6), is_breached("P2", 6))
            print("same module object:", rules is sla_rules)
            print(f"main's __name__ = {__name__!r}")
            """,
    })
    run("main.py", cwd=root)

**Explain:**

- `rules.py` ran **once**, although it was imported three times.
- Inside an imported module, `__name__` is the module's name (`'rules'`). In the file you run directly, it's `'__main__'`. That's what `if __name__ == "__main__":` tests (Session 01, Part 5): "only do this when I'm the program, not when I'm imported."
- `import rules` binds the name `rules` to the module; you then write `rules.is_breached`. `from rules import is_breached` binds the function directly. Both are normal. Prefer `import module` when a name could be ambiguous (`json.load` is clearer than a bare `load`), and `from module import name` for distinctive names you use often.
- Avoid `from rules import *`. It dumps every public name into your namespace, so neither readers nor tools can tell where a name came from.

| | Java | C# | TypeScript | Python |
|---|---|---|---|---|
| Unit of code | class in a file | class in a file | module (file) | module (file) |
| Grouping | package (folder) | namespace + assembly | folder / package | package (folder with `__init__.py`) |
| Import | `import com.acme.Rules;` | `using Acme.Rules;` | `import { isBreached } from "./rules"` | `from rules import is_breached` |
| Executes code on import? | Static initialisers, once | Static constructors, once | Yes, once | Yes, once |
| Visibility | `public`/`private` keywords | `public`/`internal`/`private` | `export` | Everything is importable; a leading `_` means "internal, please don't" |

## Part 2: Packages and the `src/` layout

A **package** is a folder of modules that you import as a unit: `incident_report.models`, `incident_report.loading`. The folder contains an `__init__.py`, which runs when the package is first imported. It's often empty, or holds a docstring and `__version__`.

For anything you'll test and share, the Python packaging community recommends the **`src/` layout**, which `uv init --package` creates:

![Anatomy of a src-layout package](../../../assets/diagrams/python-package-structure.svg)

**How this works**

- `pyproject.toml` describes the project, including `[project.scripts]`, which turns `incident_report.cli:main` into a real `incident-report` command when the package is installed.
- `src/incident_report/` is the package. Each file is a module with one responsibility, so a colleague can find, change and test one concern without reading the rest.
- `__main__.py` is what runs for `python -m incident_report`. It usually just calls `main()` from `cli.py`.
- `tests/` lives outside the package. pytest imports `incident_report` the same way any user would, from the virtual environment.
- `uv sync` installs the package in **editable** mode: the environment points back at `src/`, so your edits take effect without reinstalling.
- The point of the extra `src/` level: the project root is often on `sys.path`, and without `src/` Python could import your *source folder* instead of the installed package. Tests might pass locally and fail for users.

Let's see a minimal package work, and see what happens when Python can't find it. In a uv project, `uv sync` makes the package importable. Here we simulate that by adding `src` to the search path with the `PYTHONPATH` environment variable.

**Predict:** which of the two runs succeeds?

In [ ]:
with tempfile.TemporaryDirectory() as workdir:
    root = Path(workdir)
    write_files(root, {
        "src/ops_tools/__init__.py": '"""Small operations tools."""\n\n__version__ = "0.1.0"\n',
        "src/ops_tools/greeting.py": """
            def greet(team: str) -> str:
                return f"Good morning, {team} on-call"
            """,
        "src/ops_tools/__main__.py": """
            from ops_tools.greeting import greet

            print(greet("platform"))
            """,
    })

    print("--- without src on the path:")
    run("-m", "ops_tools", cwd=root)
    print("\n--- with src on the path (what an editable install does for you):")
    run("-m", "ops_tools", cwd=root, env_path=str(root / "src"))

**Explain:** the first run fails with `No module named ops_tools`, because nothing on `sys.path` contains it; the package lives one level down, inside `src/`. That's the `src/` layout doing its job: you can't accidentally use the package without installing it. In your project, `uv sync` installs it, and `uv run` always uses that environment.

**Absolute versus relative imports.** Inside a package you can write `from incident_report.models import Ticket` (absolute) or `from .models import Ticket` (relative, where `.` means "this package"). Both work when the package is imported properly. This course uses **absolute** imports: they read the same everywhere, and the `ImportError: attempted relative import with no known parent package` you get from running a module file directly is one less thing to debug.

**Running your code**, from most to least preferred:

| Command | When |
|---|---|
| `uv run incident-report` | The installed command, defined in `[project.scripts]` |
| `uv run python -m incident_report` | Same code, via `__main__.py`; handy when the command isn't installed |
| `uv run python src/incident_report/cli.py` | **Don't.** The file runs as a script outside its package, and its imports may fail or behave differently |

## Part 3: Classes, the Python way

You know classes. Here is what's different in Python, in one example:

In [ ]:
class SlaPolicy:
    """Response-time targets per priority."""

    escalation_contact = "duty-manager"     # class attribute: shared by all instances

    def __init__(self, hours_by_priority: dict[str, int]) -> None:
        self.hours_by_priority = dict(hours_by_priority)   # instance attribute (a copy)
        self._checks = 0                                   # leading _ = internal by convention

    def limit_for(self, priority: str) -> int:
        self._checks += 1
        return self.hours_by_priority[priority]

    @property
    def strictest(self) -> int:
        """Read like an attribute, computed like a method."""
        return min(self.hours_by_priority.values())

    def __repr__(self) -> str:
        return f"SlaPolicy({self.hours_by_priority!r})"


policy = SlaPolicy({"P1": 4, "P2": 24})
print(policy.limit_for("P2"), policy.strictest, policy.escalation_contact)
print(policy)
print(policy._checks)   # nothing stops you; the underscore is a request, not a lock

| Concept | Java / C# | Python |
|---|---|---|
| Constructor | `SlaPolicy(...) { }` | `def __init__(self, ...)` |
| The instance | implicit `this` | explicit first parameter, `self` by convention |
| Fields | declared in the class body | created by assigning `self.x = ...`, usually in `__init__` |
| Private | `private` keyword | a leading `_` by convention; nothing is enforced |
| Getter | `getStrictest()` / C# property | `@property` |
| `toString()` / `ToString()` | override | `__repr__` (for developers) and `__str__` (for users) |
| `equals()` / `Equals()` | override | `__eq__` |
| Static factory | `static SlaPolicy of(...)` | `@classmethod` (Part 4) |
| Interfaces | `interface` | duck typing: any object with the right methods works; `typing.Protocol` makes that explicit |

Methods named with double underscores (`__init__`, `__repr__`, `__eq__`) are "dunder" methods. Python calls them for you: `repr(policy)` calls `__repr__`, and `a == b` calls `__eq__`.

Writing `__init__`, `__repr__` and `__eq__` by hand for every data-holding class gets tedious, and it's easy to forget a field. That's what dataclasses solve.

## Part 4: Dataclasses

`@dataclass` generates `__init__`, `__repr__` and `__eq__` from annotated fields. It's the closest thing Python has to a Java or C# `record`, and the right default for "a bundle of named values", such as a ticket.

In [ ]:
from dataclasses import FrozenInstanceError, asdict, dataclass, field

CLOSED_STATUSES = frozenset({"resolved", "closed"})


@dataclass(frozen=True)
class Ticket:
    id: str
    team: str
    priority: str
    status: str
    age_hours: int
    assignee: str | None = None
    tags: tuple[str, ...] = ()

    @property
    def is_open(self) -> bool:
        return self.status not in CLOSED_STATUSES

    def is_breached(self, sla_hours: dict[str, int]) -> bool:
        return self.is_open and self.age_hours > sla_hours[self.priority]


ticket = Ticket(id="INC-3001", team="platform", priority="P1", status="open", age_hours=7)

print(ticket)
print(ticket.team, ticket.is_open, ticket.is_breached({"P1": 4}))
print(ticket == Ticket("INC-3001", "platform", "P1", "open", 7))
print(asdict(ticket))

What you get:

- A readable `repr`, and `==` that compares **field values**, not identity. Compare that with Session 02's `is` versus `==`.
- **Attributes instead of string keys.** `ticket.teams` is underlined by your editor and fails immediately with `AttributeError`, rather than causing a `KeyError` at 02:00. That solves the colleague's first problem.
- `asdict()` converts it back to a dict, for JSON for example.

**`frozen=True`** makes instances immutable, like Session 02's tuples. **Predict:** what happens when you try to close the ticket?

In [ ]:
try:
    ticket.status = "closed"
except FrozenInstanceError as error:
    print("FrozenInstanceError:", error)

from dataclasses import replace

closed = replace(ticket, status="closed")   # a NEW ticket with one field changed
print(ticket.status, "->", closed.status)

Immutable records are safe to share between functions, which is exactly the class of bug Sessions 02 and 03 kept hitting. When you need a changed copy, `dataclasses.replace()` builds one. (Frozen means the *fields* can't be reassigned. A frozen dataclass holding a `dict` can still have that dict's contents changed, which is why `Ticket` stores its tags as a tuple.)

### Defaults, and the mutable-default trap again

**Predict:** this class gives `tags` a list default. What happens?

In [ ]:
try:
    @dataclass
    class Draft:
        title: str
        tags: list[str] = []
except ValueError as error:
    print("ValueError:", error)


@dataclass
class Draft:
    title: str
    tags: list[str] = field(default_factory=list)   # a new list per instance


first, second = Draft("disk full"), Draft("vpn down")
first.tags.append("storage")
print(first.tags, second.tags)

**Explain:** dataclasses refuse a mutable default outright, at class-definition time, because it would be shared between instances: Session 03's trap again. `field(default_factory=list)` calls `list()` once per instance.

### Alternative constructors with `@classmethod`

Raw JSON records need cleaning before they become `Ticket`s. A **classmethod** is Python's named constructor: it receives the class (`cls`) rather than an instance, and returns a new instance.

In [ ]:
@dataclass(frozen=True)
class Ticket:
    id: str
    team: str
    priority: str
    status: str
    age_hours: int
    assignee: str | None = None
    tags: tuple[str, ...] = ()

    @classmethod
    def from_record(cls, record: dict) -> "Ticket":
        """Build a normalised Ticket from a validated raw record."""
        assignee = (record.get("assignee") or "").strip()
        return cls(
            id=record["id"],
            team=record["team"].strip().lower(),
            priority=record["priority"],
            status=record["status"].strip().lower(),
            age_hours=record["age_hours"],
            assignee=assignee or None,
            tags=tuple(sorted({tag.lower() for tag in record["tags"]})),
        )


raw = {"id": "INC-3003", "team": " network ", "priority": "P2", "status": "In_Progress",
       "age_hours": 12, "assignee": "  ", "tags": ["VPN", "network", "vpn"]}
print(Ticket.from_record(raw))

**Common Mistake:** believing the annotations validate anything. `Ticket(id=1, team=None, priority="P1", status="open", age_hours="12")` is accepted without complaint, because type hints aren't enforced (Session 03, Part 4). That's why the package validates raw records *before* calling `from_record`. Pydantic models (Session 06) are dataclass-like classes that **do** validate.

### Composition over inheritance

Coming from Java or C#, it's tempting to build hierarchies: `BaseTicket` → `IncidentTicket` → `SecurityIncidentTicket`. In Python, prefer **composition**: objects that hold other objects, and functions that work on them.

In [ ]:
from pathlib import Path


@dataclass(frozen=True)
class Config:
    input_dir: Path
    output_file: Path
    sla_hours: dict[str, int]


@dataclass(frozen=True)
class Rejection:
    file: str
    record: int | None
    id: str | None
    problems: tuple[str, ...]


@dataclass
class LoadResult:            # composition: a result HAS tickets and rejections
    tickets: list[Ticket]
    rejections: list[Rejection]

    @property
    def ok(self) -> bool:
        return not self.rejections


result = LoadResult(tickets=[Ticket.from_record(raw)], rejections=[])
print(result.ok, len(result.tickets))

| Use inheritance when... | Use composition when... |
|---|---|
| There's a genuine "is a" relationship *and* shared behaviour, e.g. `ExportError(Exception)` | One object *has* or *uses* another: a report has tickets, a loader uses a config |
| A framework requires it (exception classes, Pydantic models in Session 06, `unittest.TestCase`) | You want to swap a part, for example a different data source in tests |
| The hierarchy is shallow: one level, rarely two | You'd otherwise create a "base" class just to share a few lines |

Python's duck typing makes composition easy: a function that needs "something with `.tickets`" accepts any object with that attribute, with no interface declaration required.

## Part 5: Testing with pytest

pytest is the de facto standard Python test runner. Its conventions replace most of the ceremony you know from JUnit or xUnit:

- Test files are named `test_*.py`; test functions are named `test_*`. No classes or annotations needed.
- You check results with a plain `assert`. When it fails, pytest shows the values on both sides.
- Setup is done with **fixtures**: functions whose return values pytest passes to any test that names them as a parameter.

![What pytest does when you run it](../../../assets/diagrams/pytest-execution-flow.svg)

**How this works**

- pytest reads its settings from `[tool.pytest]` in `pyproject.toml`, then **collects** every matching file and function. An import error in a test file stops the run before any test executes, so fix those first.
- Each `parametrize` row becomes a separate test case with its own pass or fail. One function covers the boundary values that would otherwise need five near-identical tests.
- Before calling a test, pytest builds each fixture it names: your own from `conftest.py`, or built-ins such as `tmp_path`, a new empty folder for every test. After the test, it tears them down. Every test starts from a clean state.
- When an `assert` fails, pytest has rewritten it so the report shows the actual values, so you rarely need custom failure messages.
- The exit code is `0` only when everything passed, which is what CI checks.

Let's run pytest on a small project. **Predict:** how many tests does pytest report, and which fail?

In [ ]:
TICKET_TESTS = {
    "pyproject.toml": """
        [tool.pytest]
        addopts = ["-q"]
        """,
    "sla.py": """
        def is_breached(priority: str, age_hours: int, sla_hours: dict[str, int]) -> bool:
            return age_hours >= sla_hours[priority]
        """,
    "conftest.py": """
        import pytest


        @pytest.fixture
        def sla_hours():
            return {"P1": 4, "P2": 24}
        """,
    "test_sla.py": """
        import pytest

        from sla import is_breached


        @pytest.mark.parametrize(
            ("priority", "age_hours", "expected"),
            [("P1", 5, True), ("P1", 4, False), ("P2", 3, False)],
        )
        def test_is_breached(sla_hours, priority, age_hours, expected):
            assert is_breached(priority, age_hours, sla_hours) is expected


        def test_unknown_priority(sla_hours):
            with pytest.raises(KeyError, match="P9"):
                is_breached("P9", 1, sla_hours)
        """,
}

with tempfile.TemporaryDirectory() as workdir:
    root = Path(workdir)
    write_files(root, TICKET_TESTS)
    exit_code = run("-m", "pytest", cwd=root)
    print("exit code:", exit_code)

**Explain:** four test cases: three from `parametrize`, plus `test_unknown_priority`. The `("P1", 4, False)` case fails: the function uses `>=`, so a ticket exactly at its limit counts as breached. Read the failure report:

- the test ID, `test_is_breached[P1-4-False]`, names the exact parametrize row;
- the `assert` line shows the actual values: `assert True is False`, with `True` being the function's result;
- the exit code is `1`.

**Modify:** in `TICKET_TESTS`, change `>=` to `>` in `sla.py` and re-run. All four should pass, with exit code `0`.

### The techniques you'll use most

| Need | pytest | JUnit 5 | xUnit | Jest |
|---|---|---|---|---|
| A test | `def test_x():` | `@Test void x()` | `[Fact] void X()` | `test("x", () => ...)` |
| Check a value | `assert a == b` | `assertEquals(b, a)` | `Assert.Equal(b, a)` | `expect(a).toBe(b)` |
| Expect an exception | `with pytest.raises(KeyError, match="P9"):` | `assertThrows(...)` | `Assert.Throws<...>(...)` | `expect(fn).toThrow(...)` |
| Same test, many inputs | `@pytest.mark.parametrize(...)` | `@ParameterizedTest` | `[Theory]` + `[InlineData]` | `test.each(...)` |
| Shared setup | a fixture, by parameter name | `@BeforeEach` | constructor / fixture class | `beforeEach` |
| A temporary folder | `tmp_path` fixture | `@TempDir` | (manual) | (manual) |
| Captured output | `capsys` fixture | (manual) | `ITestOutputHelper` | spy on `console` |

### Writing good tests

- **Arrange, act, assert.** Build the input, call one thing, check the result. Blank lines between the three make tests easy to scan.
- **Test behaviour, not implementation.** Assert on what a function returns or writes, not on which helpers it calls internally, so you can refactor freely.
- **Name the case.** `test_breach_at_exactly_the_limit_is_not_a_breach` documents a business rule; `test_breach_2` doesn't.
- **Test the edges.** Empty input, exactly-at-the-limit values, duplicates, `None`, invalid files. Most bugs in Sessions 02 and 03 lived at the edges.
- **Keep tests independent.** Each test builds its own data (fixtures make that cheap). Tests that share mutable state pass alone and fail together, which the debugging lab demonstrates.
- **Keep them fast and offline.** Use `tmp_path` for files. Mock external services, as you'll do for HTTP in Session 05 and for LLMs in Session 09.

## Part 6: Ruff for the whole project

You've run Ruff with its defaults since Session 01. In a package, you configure it once in `pyproject.toml`, so every developer and the CI pipeline apply the same rules:

```toml
[tool.ruff]
line-length = 100

[tool.ruff.lint]
select = ["E", "F", "I", "B", "UP", "SIM"]
```

| Code | Rule family | Catches, for example |
|---|---|---|
| `E` | pycodestyle errors | Overlong lines, bad comparisons such as `== None` |
| `F` | Pyflakes | Unused imports and variables, undefined names |
| `I` | isort | Imports out of order |
| `B` | flake8-bugbear | Likely bugs: mutable defaults (`B006`), loop-variable closures |
| `UP` | pyupgrade | Outdated syntax, such as `Optional[int]` where the newer union syntax works |
| `SIM` | flake8-simplify | Needlessly complicated code |

**Predict:** which problems will Ruff report in this file?

In [ ]:
RUFF_DEMO = {
    "pyproject.toml": """
        [tool.ruff]
        line-length = 100

        [tool.ruff.lint]
        select = ["E", "F", "I", "B", "UP", "SIM"]
        """,
    "report.py": """
        import sys
        import json
        from typing import Optional


        def collect(ticket_ids, seen=[]):
            for ticket_id in ticket_ids:
                if ticket_id in seen:
                    pass
                else:
                    seen.append(ticket_id)
            return seen


        def owner(ticket) -> Optional[str]:
            if ticket.get("assignee") == None:
                return None
            return ticket["assignee"]


        print(json.dumps(collect(["INC-1"])))
        """,
}

with tempfile.TemporaryDirectory() as workdir:
    root = Path(workdir)
    write_files(root, RUFF_DEMO)
    run("-m", "ruff", "check", "--output-format", "concise", "--no-cache", ".", cwd=root)

**Explain** each finding before you look up its rule. `uv run ruff check --fix .` fixes the safe ones automatically (import order, `Optional`), and `uv run ruff format .` handles layout. The mutable default (`B006`) needs your judgement, which is why it isn't auto-fixed.

To silence one finding on purpose, add `# noqa: <code>` to that line, with a reason, as the Session 02 and 03 checkers did for `BLE001`. If you find yourself doing that often, the rule probably doesn't fit the project: change the configuration instead.

## Part 7: Debugging with breakpoints

Print statements get you surprisingly far, but a debugger lets you stop at a line and inspect *everything*. There are two ways to stop.

### `breakpoint()` in any terminal

Add `breakpoint()` on the line before the one you suspect, and run the program or test normally:

```python
def team_summary(tickets, sla_hours):
    open_by_team = {ticket.team: [] for ticket in tickets}
    breakpoint()          # execution pauses here, in the pdb debugger
    ...
```

```bash
uv run incident-report                          # or:
uv run pytest tests/unit/test_summary.py -s     # -s lets pdb use your terminal
```

At the `(Pdb)` prompt:

| Command | Does |
|---|---|
| `p open_by_team` | Print an expression |
| `pp tickets[0]` | Pretty-print it |
| `l` | Show the source around the current line |
| `n` | Run the **n**ext line, stepping over calls |
| `s` | **S**tep into the call on this line |
| `c` | **C**ontinue until the next breakpoint |
| `w` | Show **w**here you are: the call stack, like a traceback |
| `q` | **Q**uit |

Even better for failing tests: `uv run pytest --pdb` opens the debugger **at the moment a test fails**, with every variable intact. Remove every `breakpoint()` before you commit; Ruff's `T100` rule can catch forgotten ones.

### The VS Code debugger

1. Click in the gutter left of a line number to set a breakpoint (a red dot).
2. Open the **Testing** view (the flask icon), find a test, and choose **Debug Test**. Or open `cli.py` and run **Python Debugger: Debug Python File**.
3. When execution pauses, the **Variables** panel shows every local variable, **Watch** evaluates expressions, and the **Call Stack** panel is a live traceback you can click through.
4. Step with F10 (over), F11 (into) and Shift+F11 (out); continue with F5.

VS Code must use the project's `.venv` interpreter (Session 01, Part 6), or it won't find `incident_report` or pytest. The workbook asks you to use both techniques on your own code.

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| Naming your file `json.py`, `logging.py`, `test.py`... | Shadows the standard-library module | Use distinctive names; check `module.__file__` when imports behave oddly |
| Two modules importing each other | `ImportError: cannot import name ... (most likely due to a circular import)` | Make dependencies point one way; move shared code into a lower module |
| Running `python src/pkg/module.py` | Imports fail, or relative imports raise `ImportError` | `uv run python -m pkg` or the installed command |
| `from module import *` | Unclear where names come from | Import names explicitly |
| Deep inheritance hierarchies | Rigid code; changes ripple through subclasses | Dataclasses plus composition |
| A mutable default in a dataclass | `ValueError` at class definition | `field(default_factory=list)` |
| Expecting dataclass hints to validate | Wrong types accepted silently | Validate first (or use Pydantic, Session 06) |
| Tests that share mutable data | Pass alone, fail together | Fresh data from a fixture in every test |
| `assert (a == b, "message")` | Always true: a non-empty tuple is truthy | `assert a == b, "message"` |
| Duplicate test file names in different folders | `import file mismatch` collection errors | `--import-mode=importlib` (set in the project's `addopts`) or unique names |
| Leaving `breakpoint()` in code | CI hangs or fails waiting for input | Remove it; let Ruff check for it |

## Practical engineering application

Every project from here on uses this structure. Session 05's API client becomes a module with tests that mock HTTP. Session 07's FastAPI service is a package with routers in separate modules. Session 09's LLM integration sits behind a small class that tests replace with a fake. The habits are the same each time: **one responsibility per module, dependencies pointing one way, immutable data objects, and fast tests that run on every change.**

## Key takeaways

- A module is a file and a package is a folder of modules. Imports search `sys.path` in order and run each module once, caching it in `sys.modules`.
- Use the `src/` layout, absolute imports, `__main__.py` for `python -m`, and `[project.scripts]` for a command. `uv sync` installs the package in editable mode.
- Python classes have explicit `self`, no enforced privacy, and dunder methods. For data, use `@dataclass`, usually `frozen=True`, with `@classmethod` constructors.
- Prefer composition to inheritance; reserve inheritance for genuine "is a" cases such as exceptions.
- pytest: `test_*` functions, plain `assert`, fixtures by parameter name, `tmp_path`, `parametrize`, `pytest.raises`. Keep tests independent, fast and focused on behaviour.
- Configure Ruff once in `pyproject.toml`. Debug with `breakpoint()`, `pytest --pdb` or VS Code.

## Further reading

The curated references for this session are in [`../resources/references.md`](../resources/references.md).

**Next:** work through **`03-debugging-lab.ipynb`**, then test yourself with **`02-practice-quiz.ipynb`**.